# 1. Configura JWT Authentication

Mismos casos de uso que en `3_VSO.ipynb`: secreto estático, actualizaciones, secreto dinámico de base de datos y certificado de PKI, sincronizados con el Vault Secrets Operator.

La diferencia es el login. Vault no llama a TokenReview. Valida el JWT de la ServiceAccount con las claves públicas que publica el API server de minikube.

In [47]:
%env WORKDIR=/tmp/vault
%env VAULT_K8S_NAMESPACE=vault
%env VAULT_HELM_RELEASE_NAME=vault
%env VAULT_SERVICE_NAME=vault-internal
%env K8S_CLUSTER_NAME=cluster.local

env: WORKDIR=/tmp/vault
env: VAULT_K8S_NAMESPACE=vault
env: VAULT_HELM_RELEASE_NAME=vault
env: VAULT_SERVICE_NAME=vault-internal
env: K8S_CLUSTER_NAME=cluster.local


El root token, la dirección y la CA de Vault están en `${WORKDIR}/config.env`, generado al desplegar Vault. Las celdas que ejecutan `vault` cargan ese fichero con `source` para no dejar el token escrito en el cuaderno.

## Descubrimiento OIDC

El API server publica el emisor y el JWKS en `/.well-known/openid-configuration` y `/openid/v1/jwks`. Esos endpoints solo devuelven claves públicas. Vault los consulta desde el pod, sin token de Kubernetes, así que hay que permitir el acceso anónimo.

In [48]:
%%bash
# ClusterRole system:service-account-issuer-discovery para system:unauthenticated.
kubectl create clusterrolebinding oidc-reviewer \
  --clusterrole=system:service-account-issuer-discovery \
  --group=system:unauthenticated

clusterrolebinding.rbac.authorization.k8s.io/oidc-reviewer created


In [49]:
%%bash
kubectl get --raw /.well-known/openid-configuration | jq

{
  "issuer": "https://kubernetes.default.svc.cluster.local",
  "jwks_uri": "https://192.168.49.2:8443/openid/v1/jwks",
  "response_types_supported": [
    "id_token"
  ],
  "subject_types_supported": [
    "public"
  ],
  "id_token_signing_alg_values_supported": [
    "RS256"
  ]
}


`issuer` es el claim `iss` del token. En minikube es `https://kubernetes.default.svc.cluster.local`.

`jwks_uri` apunta a la IP del nodo. Esa IP cambia al recrear minikube. Vault corre dentro del clúster, así que usamos el Service `kubernetes.default.svc.cluster.local` y el path de `jwks_uri` (`/openid/v1/jwks`).

## CA del API server

In [50]:
%%bash
# La misma CA que Kubernetes monta en los pods. Vault verifica con ella el TLS del JWKS.
kubectl exec -n ${VAULT_K8S_NAMESPACE} vault-0 -- \
  cat /var/run/secrets/kubernetes.io/serviceaccount/ca.crt > ${WORKDIR}/k8s-ca.pem

openssl x509 -noout -subject -in ${WORKDIR}/k8s-ca.pem

subject= /CN=minikubeCA


In [51]:
%%bash
# Lectura anónima desde el pod de Vault.
# --no-check-certificate es solo para esta comprobación; la config de Vault sí usa la CA.
kubectl exec -n ${VAULT_K8S_NAMESPACE} vault-0 -- \
  wget -q -O- --no-check-certificate \
  https://kubernetes.default.svc.cluster.local/openid/v1/jwks \
  | jq -r '.keys[].kid'

0l5R0WBr-Im3U3ZeevttknJlHuUmiRGbbkn3A8Xq40w


## Habilitar autenticación JWT

`jwks_url` hace que Vault descargue las claves públicas del clúster al validar un token. Si minikube rota la clave de firma, no hay que volver a copiarla.


In [52]:
%%bash
set -a; source ${WORKDIR}/config.env; set +a
vault auth enable jwt

Success! Enabled jwt auth method at: jwt/


In [53]:
%%bash
set -a; source ${WORKDIR}/config.env; set +a

JWKS_PATH=$(kubectl get --raw /.well-known/openid-configuration | jq -r '.jwks_uri' | sed 's|https://[^/]*||')
JWKS_URL="https://kubernetes.default.svc.cluster.local${JWKS_PATH}"
echo "JWKS_URL: ${JWKS_URL}"

vault write auth/jwt/config \
  jwks_url="${JWKS_URL}" \
  jwks_ca_pem=@${WORKDIR}/k8s-ca.pem

JWKS_URL: https://kubernetes.default.svc.cluster.local/openid/v1/jwks
Success! Data written to: auth/jwt/config


## Rol

El rol admite las namespaces `static-jwt`, `static`, `database`, `pki`, `agent`, `test`, `default` y `csi`. El claim `sub` tiene la forma `system:serviceaccount:<namespace>:<nombre>`.

In [54]:
%%bash

set -a; source ${WORKDIR}/config.env; set +a

ISSUER=$(kubectl get --raw /.well-known/openid-configuration | jq -r '.issuer')
echo "bound_audiences: ${ISSUER}"

cat > ${WORKDIR}/jwt-role.json <<EOF
{
  "role_type": "jwt",
  "bound_audiences": "${ISSUER}",
  "user_claim": "sub",
  "bound_claims_type": "glob",
  "bound_claims": {
    "sub": [
      "system:serviceaccount:static-jwt:*",
      "system:serviceaccount:static:*",
      "system:serviceaccount:database:*",
      "system:serviceaccount:pki:*",
      "system:serviceaccount:agent:*",
      "system:serviceaccount:test:*",
      "system:serviceaccount:default:*",
      "system:serviceaccount:csi:*"
    ]
  },
  "policies": ["devk8s"],
  "ttl": "30m"
}
EOF

vault write auth/jwt/role/role @${WORKDIR}/jwt-role.json

bound_audiences: https://kubernetes.default.svc.cluster.local
Success! Data written to: auth/jwt/role/role


Y la política asociada al role que hemos creado en el paso previos

In [55]:
%%bash
set -a; source ${WORKDIR}/config.env; set +a

vault policy write devk8s - <<EOF
path "kvv2/*" {
  capabilities = ["read"]
}
path "kv_v2/*" {
  capabilities = ["read"]
}
EOF

Success! Uploaded policy: devk8s


### Creamos un secreto estático en Vault

Habilitamos la engine

In [56]:
%%bash
set -a; source ${WORKDIR}/config.env; set +a

vault secrets enable -path=kv_v2 kv-v2

Success! Enabled the kv-v2 secrets engine at: kv_v2/


Creamos un secreto

In [57]:
%%bash
set -a; source ${WORKDIR}/config.env; set +a

vault kv put kv_v2/webapp/config username="static-user" password="static-password"

====== Secret Path ======
kv_v2/data/webapp/config

======= Metadata =======
Key                Value
---                -----
created_time       2026-10-07T16:14:54.3232014Z
custom_metadata    <nil>
deletion_time      n/a
destroyed          false
version            1


# 2. Instalando el VSO usando helm. [Referencia](https://developer.hashicorp.com/vault/docs/platform/k8s/vso/installation)

In [60]:
%%bash
helm repo add hashicorp https://helm.releases.hashicorp.com
helm repo update
# Los recursos se instalaran en el namespace vault-secrets-operator
helm install --version 1.0.1 --create-namespace --namespace vault-secrets-operator vault-secrets-operator hashicorp/vault-secrets-operator

"hashicorp" already exists with the same configuration, skipping
Hang tight while we grab the latest from your chart repositories...
...Successfully got an update from the "secrets-store-csi-driver" chart repository
...Successfully got an update from the "eks" chart repository
...Successfully got an update from the "kspm-helm-charts" chart repository
...Successfully got an update from the "hashicorp" chart repository
...Successfully got an update from the "prometheus-community" chart repository
...Successfully got an update from the "bitnami" chart repository
Update Complete. ⎈Happy Helming!⎈
NAME: vault-secrets-operator
LAST DEPLOYED: Wed Oct  7 18:15:12 2026
NAMESPACE: vault-secrets-operator
STATUS: deployed
REVISION: 1
DESCRIPTION: Install complete


In [61]:
! kubectl get events -n vault-secrets-operator

LAST SEEN   TYPE     REASON              OBJECT                                                            MESSAGE
9s          Normal   LeaderElection      lease/b0d477c0.hashicorp.com                                      vault-secrets-operator-controller-manager-64b7c5d64c-s9mnv_03eb7409-ab8c-43a3-9153-b8fbdc8a0ade became leader
9s          Normal   Scheduled           pod/vault-secrets-operator-controller-manager-64b7c5d64c-s9mnv    Successfully assigned vault-secrets-operator/vault-secrets-operator-controller-manager-64b7c5d64c-s9mnv to workshop
9s          Normal   Pulled              pod/vault-secrets-operator-controller-manager-64b7c5d64c-s9mnv    Container image "quay.io/brancz/kube-rbac-proxy:v0.18.1" already present on machine and can be accessed by the pod
9s          Normal   Created             pod/vault-secrets-operator-controller-manager-64b7c5d64c-s9mnv    Container created
9s          Normal   Started             pod/vault-secrets-operator-controller-manager-64b7c5d64c-

In [63]:
! kubectl get pods -n vault-secrets-operator

NAME                                                         READY   STATUS    RESTARTS   AGE
vault-secrets-operator-controller-manager-64b7c5d64c-s9mnv   2/2     Running   0          21s


### Instalamos los CRDs

[VaultConnection](https://developer.hashicorp.com/vault/docs/platform/k8s/vso/api-reference#vaultconnectionspec)

In [64]:
%%bash

cat > ${WORKDIR}/vso_crd.yaml <<EOF
---
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultConnection
metadata:
  namespace: vault-secrets-operator
  name: example
spec:
  address: https://vault.vault.svc.cluster.local:8200
  skipTLSVerify: true

EOF

kubectl apply -f ${WORKDIR}/vso_crd.yaml

vaultconnection.secrets.hashicorp.com/example created


Verify VaultConnection is being deployed

In [65]:
! kubectl describe VaultConnection example -n vault-secrets-operator

Name:         example
Namespace:    vault-secrets-operator
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultConnection
Metadata:
  Creation Timestamp:  2026-10-07T16:15:35Z
  Finalizers:
    vaultconnection.secrets.hashicorp.com/finalizer
  Generation:        1
  Resource Version:  21858
  UID:               24bf8483-27fe-4207-904c-9e72226d58c0
Spec:
  Address:         https://vault.vault.svc.cluster.local:8200
  Skip TLSVerify:  true
Status:
  Conditions:
    Last Transition Time:  2026-10-07T16:15:35Z
    Message:               Vault ping, address=https://vault.vault.svc.cluster.local:8200
    Observed Generation:   1
    Reason:                Accepted
    Status:                True
    Type:                  VaultPing
    Last Transition Time:  2026-10-07T16:15:35Z
    Message:               Successfully validated resource, address=https://vault.vault.svc.cluster.local:8200
    Observed Generation:   1
    Reason:            

[VaultAuth](https://developer.hashicorp.com/vault/docs/platform/k8s/vso/api-reference#vaultauthspec)

VSO pide un JWT de la ServiceAccount `default` en el namespace del secreto, con la audiencia del emisor del clúster, y lo envía a `auth/jwt/login`.

In [66]:
%%bash
ISSUER=$(kubectl get --raw /.well-known/openid-configuration | jq -r '.issuer')

cat > ${WORKDIR}/vaultauth_crd.yaml <<EOF
---
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultAuth
metadata:
  namespace: vault-secrets-operator
  name: example-jwt
spec:
  vaultConnectionRef: example
  allowedNamespaces: ["*"]
  method: jwt
  mount: jwt

  jwt:
    role: role
    serviceAccount: default
    audiences: ["${ISSUER}"]

EOF
kubectl apply -f ${WORKDIR}/vaultauth_crd.yaml

vaultauth.secrets.hashicorp.com/example-jwt created


Verify VaultAuth is being deployed

In [67]:
! kubectl describe VaultAuth example-jwt -n vault-secrets-operator

Name:         example-jwt
Namespace:    vault-secrets-operator
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultAuth
Metadata:
  Creation Timestamp:  2026-10-07T16:15:38Z
  Finalizers:
    vaultauth.secrets.hashicorp.com/finalizer
  Generation:        1
  Resource Version:  21864
  UID:               82204d5d-ea94-46d7-ac51-f0b97fcdc397
Spec:
  Allowed Namespaces:
    *
  Jwt:
    Audiences:
      https://kubernetes.default.svc.cluster.local
    Role:                      role
    Service Account:           default
    Token Expiration Seconds:  600
  Method:                      jwt
  Mount:                       jwt
  Vault Connection Ref:        example
Status:
  Conditions:
    Last Transition Time:  2026-10-07T16:15:38Z
    Message:               VaultAuthHealthy
    Observed Generation:   1
    Reason:                Healthy
    Status:                True
    Type:                  Healthy
  Spec Hash:               d504270

[VaultStaticSecret](https://developer.hashicorp.com/vault/docs/platform/k8s/vso/api-reference#vaultstaticsecretspec)


In [68]:
! kubectl create ns static-jwt

namespace/static-jwt created


In [69]:
%%bash
## Support KVv1 and KVv2
cat > ${WORKDIR}/static_secret.yaml <<EOF
---
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultStaticSecret
metadata:
  namespace: static-jwt
  name: example
spec:
  vaultAuthRef: vault-secrets-operator/example-jwt
  mount: kv_v2
  type: kv-v2
  path: webapp/config
  refreshAfter: 60s
  destination:
    create: true
    name: static-secret1
EOF

kubectl apply -f ${WORKDIR}/static_secret.yaml

vaultstaticsecret.secrets.hashicorp.com/example created


Verificamos que el secreto se sincroniza

In [70]:
! kubectl describe VaultStaticSecret example -n static-jwt

Name:         example
Namespace:    static-jwt
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultStaticSecret
Metadata:
  Creation Timestamp:  2026-10-07T16:15:44Z
  Finalizers:
    vaultstaticsecret.secrets.hashicorp.com/finalizer
  Generation:        2
  Resource Version:  21881
  UID:               40a2978e-226b-4104-8d8e-02f315b7d765
Spec:
  Destination:
    Create:     true
    Name:       static-secret1
    Overwrite:  false
    Transformation:
  Hmac Secret Data:  true
  Mount:             kv_v2
  Path:              webapp/config
  Refresh After:     60s
  Type:              kv-v2
  Vault Auth Ref:    vault-secrets-operator/example-jwt
Status:
  Conditions:
    Last Transition Time:  2026-10-07T16:15:44Z
    Message:               Secret synced, horizon=51.548650282s
    Observed Generation:   2
    Reason:                Synced
    Status:                True
    Type:                  SecretSynced
    Last Transition Time:

Por último veriquemos que podemos leer el secreto (decodificar el base64)

In [73]:
%%bash
echo "USERNAME: $(kubectl get secret static-secret1 -n static-jwt -o json | jq -r .data.username | base64 -d)"
echo "PASSWORD: $(kubectl get secret static-secret1 -n static-jwt -o json | jq -r .data.password | base64 -d)"

USERNAME: static-user
PASSWORD: static-password


In [75]:
%%bash
cat > ${WORKDIR}/mypod.yaml <<EOF
apiVersion: v1
kind: Pod
metadata:
  name: mypod
  namespace: static-jwt
spec:
  containers:
  - name: mypod
    image: nginx
    env:
    - name: USERNAME
      valueFrom:
        secretKeyRef:
          name: static-secret1
          key: username
    - name: PASSWORD
      valueFrom:
        secretKeyRef:
          name: static-secret1
          key: password
EOF

kubectl apply -f ${WORKDIR}/mypod.yaml
sleep 10



pod/mypod created


In [76]:
! kubectl exec mypod -n static-jwt -- env | grep -E 'USERNAME|PASSWORD'

USERNAME=static-user
PASSWORD=static-password


## Rollout Restart

In [78]:
%%bash
## Support KVv1 and KVv2
cat > ${WORKDIR}/static_secret2.yaml <<EOF
---
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultStaticSecret
metadata:
  namespace: static-jwt
  name: example2
spec:
  vaultAuthRef: vault-secrets-operator/example-jwt
  mount: kv_v2
  type: kv-v2
  hmacSecretData: true          # Enable HMAC signing of secret data. Verify changes to secret data.
  rolloutRestartTargets:        # Restart target deployments when secret data changes.
    - kind: Deployment
      name: mydeployment
  path: webapp/config
  refreshAfter: 60s             # Check for updates every 60 seconds. Not using events
  destination:
    create: true
    name: static-secret2
EOF

kubectl apply -f ${WORKDIR}/static_secret2.yaml

vaultstaticsecret.secrets.hashicorp.com/example2 created


In [79]:
! kubectl describe VaultStaticSecret example2 -n static-jwt

Name:         example2
Namespace:    static-jwt
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultStaticSecret
Metadata:
  Creation Timestamp:  2026-10-07T16:16:44Z
  Finalizers:
    vaultstaticsecret.secrets.hashicorp.com/finalizer
  Generation:        2
  Resource Version:  21979
  UID:               7a85164a-486c-4907-8f30-7e996409f8d2
Spec:
  Destination:
    Create:     true
    Name:       static-secret2
    Overwrite:  false
    Transformation:
  Hmac Secret Data:  true
  Mount:             kv_v2
  Path:              webapp/config
  Refresh After:     60s
  Rollout Restart Targets:
    Kind:          Deployment
    Name:          mydeployment
  Type:            kv-v2
  Vault Auth Ref:  vault-secrets-operator/example-jwt
Status:
  Conditions:
    Last Transition Time:  2026-10-07T16:16:44Z
    Message:               Secret synced, horizon=51.723466822s
    Observed Generation:   2
    Reason:                Synced
    Status:

In [80]:
%%bash
cat > ${WORKDIR}/mydeployment.yaml <<EOF
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mydeployment
  namespace: static-jwt
spec:
  replicas: 2
  selector:
    matchLabels:
      app: myapp
  template:
    metadata:
      labels:
        app: myapp
    spec:
      containers:
      - name: mypod
        image: nginx
        env:
        - name: USERNAME
          valueFrom:
            secretKeyRef:
              name: static-secret2
              key: username
        - name: PASSWORD
          valueFrom:
            secretKeyRef:
              name: static-secret2
              key: password
EOF

kubectl apply -f ${WORKDIR}/mydeployment.yaml

deployment.apps/mydeployment created


In [83]:
! kubectl get deployments -n static-jwt

NAME           READY   UP-TO-DATE   AVAILABLE   AGE
mydeployment   2/2     2            2           11s


In [84]:
! kubectl get deployment mydeployment -n static-jwt -o yaml

apiVersion: apps/v1
kind: Deployment
metadata:
  annotations:
    deployment.kubernetes.io/revision: "1"
    kubectl.kubernetes.io/last-applied-configuration: |
      {"apiVersion":"apps/v1","kind":"Deployment","metadata":{"annotations":{},"name":"mydeployment","namespace":"static-jwt"},"spec":{"replicas":2,"selector":{"matchLabels":{"app":"myapp"}},"template":{"metadata":{"labels":{"app":"myapp"}},"spec":{"containers":[{"env":[{"name":"USERNAME","valueFrom":{"secretKeyRef":{"key":"username","name":"static-secret2"}}},{"name":"PASSWORD","valueFrom":{"secretKeyRef":{"key":"password","name":"static-secret2"}}}],"image":"nginx","name":"mypod"}]}}}}
  creationTimestamp: "2026-10-07T16:17:23Z"
  generation: 1
  name: mydeployment
  namespace: static-jwt
  resourceVersion: "22066"
  uid: 6e31c711-cea2-470a-b308-ef170bebf96c
spec:
  progressDeadlineSeconds: 600
  replicas: 2
  revisionHistoryLimit: 10
  selector:
    matchLabels:
      app: myapp
  strategy:
    rollingUpdate:
      maxSurge:

In [85]:
! kubectl exec deploy/mydeployment -n static-jwt -- env | grep -E 'USERNAME|PASSWORD'

USERNAME=static-user
PASSWORD=static-password


In [86]:
%%bash
set -a; source ${WORKDIR}/config.env; set +a

vault kv put kv_v2/webapp/config username="static-user2" password="static-password2"

====== Secret Path ======
kv_v2/data/webapp/config

======= Metadata =======
Key                Value
---                -----
created_time       2026-10-07T16:18:03.941134282Z
custom_metadata    <nil>
deletion_time      n/a
destroyed          false
version            2


In [90]:
! kubectl describe deployment mydeployment -n static-jwt

Name:                   mydeployment
Namespace:              static-jwt
CreationTimestamp:      Wed, 07 Oct 2026 18:17:23 +0200
Labels:                 <none>
Annotations:            deployment.kubernetes.io/revision: 2
Selector:               app=myapp
Replicas:               2 desired | 2 updated | 2 total | 2 available | 0 unavailable
StrategyType:           RollingUpdate
MinReadySeconds:        0
RollingUpdateStrategy:  25% max unavailable, 25% max surge
Pod Template:
  Labels:       app=myapp
  Annotations:  vso.secrets.hashicorp.com/restartedAt: 2026-10-07T16:18:27Z
  Containers:
   mypod:
    Image:      nginx
    Port:       <none>
    Host Port:  <none>
    Environment:
      USERNAME:    <set to the key 'username' in secret 'static-secret2'>  Optional: false
      PASSWORD:    <set to the key 'password' in secret 'static-secret2'>  Optional: false
    Mounts:        <none>
  Volumes:         <none>
  Node-Selectors:  <none>
  Tolerations:     <none>
Conditions:
  Type        

In [88]:
! kubectl get rs -n static-jwt -l app=myapp

NAME                      DESIRED   CURRENT   READY   AGE
mydeployment-58f898994b   2         2         2       58s


In [89]:
! kubectl get pods -n static-jwt -l app=myapp

NAME                            READY   STATUS    RESTARTS   AGE
mydeployment-58f898994b-2rn65   1/1     Running   0          63s
mydeployment-58f898994b-qcn56   1/1     Running   0          63s


## Instant updates for VaultStaticSecrets

### Set Event permissions
https://developer.hashicorp.com/vault/docs/concepts/events#policies

In [91]:
%%bash
set -a; source ${WORKDIR}/config.env; set +a


vault policy write devk8s - <<EOF
path "kvv2/*" {
  capabilities = ["read", "list", "subscribe"]
  subscribe_event_types = ["*"]                     # https://developer.hashicorp.com/vault/docs/concepts/events#event-types
}
path "kv_v2/*" {
  capabilities = ["read", "list", "subscribe"]
  subscribe_event_types = ["*"]
}
path "sys/events/subscribe/*" {
  capabilities = ["read"]
}
EOF

Success! Uploaded policy: devk8s


### Create VaultStaticSecret
https://developer.hashicorp.com/vault/docs/deploy/kubernetes/vso/sources/vault/instant-updates#step-2-enable-instant-updates-on-the-vaultstaticsecret

In [92]:
%%bash
## Support KVv1 and KVv2
cat > ${WORKDIR}/static_secret_event.yaml <<EOF
---
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultStaticSecret
metadata:
  namespace: static-jwt
  name: example-event
spec:
  vaultAuthRef: vault-secrets-operator/example-jwt
  mount: kv_v2
  type: kv-v2
  path: webapp/config
  refreshAfter: 1h
  destination:
    create: true
    name: static-secret-event
  syncConfig:
    instantUpdates: true
EOF

kubectl apply -f ${WORKDIR}/static_secret_event.yaml

vaultstaticsecret.secrets.hashicorp.com/example-event created


In [93]:
! kubectl describe VaultStaticSecret example-event -n static-jwt

Name:         example-event
Namespace:    static-jwt
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultStaticSecret
Metadata:
  Creation Timestamp:  2026-10-07T16:19:07Z
  Finalizers:
    vaultstaticsecret.secrets.hashicorp.com/finalizer
  Generation:        2
  Resource Version:  22279
  UID:               d6ff503b-f83b-4f83-b22e-4608d6a9ba81
Spec:
  Destination:
    Create:     true
    Name:       static-secret-event
    Overwrite:  false
    Transformation:
  Hmac Secret Data:  true
  Mount:             kv_v2
  Path:              webapp/config
  Refresh After:     1h
  Sync Config:
    Instant Updates:  true
  Type:               kv-v2
  Vault Auth Ref:     vault-secrets-operator/example-jwt
Status:
  Conditions:
    Last Transition Time:  2026-10-07T16:19:07Z
    Message:               Secret synced, horizon=49m11.668524651s
    Observed Generation:   2
    Reason:                Synced
    Status:                True
    Type

In [94]:
%%bash
set -a; source ${WORKDIR}/config.env; set +a

vault kv put kv_v2/webapp/config username="static-user13" password="static-password13"

====== Secret Path ======
kv_v2/data/webapp/config

======= Metadata =======
Key                Value
---                -----
created_time       2026-10-07T16:19:20.67793975Z
custom_metadata    <nil>
deletion_time      n/a
destroyed          false
version            3


In [95]:
! kubectl describe VaultStaticSecret example-event -n static-jwt

Name:         example-event
Namespace:    static-jwt
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultStaticSecret
Metadata:
  Creation Timestamp:  2026-10-07T16:19:07Z
  Finalizers:
    vaultstaticsecret.secrets.hashicorp.com/finalizer
  Generation:        2
  Resource Version:  22301
  UID:               d6ff503b-f83b-4f83-b22e-4608d6a9ba81
Spec:
  Destination:
    Create:     true
    Name:       static-secret-event
    Overwrite:  false
    Transformation:
  Hmac Secret Data:  true
  Mount:             kv_v2
  Path:              webapp/config
  Refresh After:     1h
  Sync Config:
    Instant Updates:  true
  Type:               kv-v2
  Vault Auth Ref:     vault-secrets-operator/example-jwt
Status:
  Conditions:
    Last Transition Time:  2026-10-07T16:19:20Z
    Message:               Secret synced, horizon=51m1.684884615s
    Observed Generation:   2
    Reason:                Synced
    Status:                True
    Type:

In [96]:
%%bash
kubectl get secret static-secret-event -n static-jwt -o json | jq -r .data.username | base64 -d
echo ""
kubectl get secret static-secret-event -n static-jwt -o json | jq -r .data.password | base64 -d

static-user13
static-password13

In [97]:
%%bash
cat > ${WORKDIR}/mydeployment2.yaml <<EOF
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mydeployment2
  namespace: static-jwt
spec:
  replicas: 2
  selector:
    matchLabels:
      app: myapp2
  template:
    metadata:
      labels:
        app: myapp2
    spec:
      containers:
      - name: mypod
        image: nginx
        env:
        - name: USERNAME
          valueFrom:
            secretKeyRef:
              name: static-secret-event
              key: username
        - name: PASSWORD
          valueFrom:
            secretKeyRef:
              name: static-secret-event
              key: password
EOF

kubectl apply -f ${WORKDIR}/mydeployment2.yaml

deployment.apps/mydeployment2 created


### Dynamic Secrets

Actualizar la políticas con permisos de lectura al path de la database

In [98]:
%%bash
set -a; source ${WORKDIR}/config.env; set +a


vault policy write devk8s - <<EOF
path "kvv2/*" {
  capabilities = ["read", "list", "subscribe"]
  subscribe_event_types = ["*"]                     # https://developer.hashicorp.com/vault/docs/concepts/events#event-types
}
path "kv_v2/*" {
  capabilities = ["read", "list", "subscribe"]
  subscribe_event_types = ["*"]
}
path "sys/events/subscribe/*" {
  capabilities = ["read"]
}
path "database/creds/readonly" {
  capabilities = [ "read"]
}
EOF

Success! Uploaded policy: devk8s


[VaultDynamicSecret](https://developer.hashicorp.com/vault/docs/platform/k8s/vso/api-reference#vaultdynamicsecretspec)

In [99]:
! kubectl create ns database

namespace/database created


In [100]:
%%bash

cat > ${WORKDIR}/dynamic_secret.yaml <<EOF
---
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultDynamicSecret
metadata:
  namespace: database
  name: db-secret
spec:
  vaultAuthRef: vault-secrets-operator/example-jwt
  mount: database
  path: creds/readonly
  destination:
    create: true
    name: db-secret
EOF

kubectl apply -f ${WORKDIR}/dynamic_secret.yaml


vaultdynamicsecret.secrets.hashicorp.com/db-secret created


In [109]:
! kubectl describe VaultDynamicSecret db-secret -n database

Name:         db-secret
Namespace:    database
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultDynamicSecret
Metadata:
  Creation Timestamp:  2026-10-07T16:24:48Z
  Finalizers:
    vaultdynamicsecret.secrets.hashicorp.com/finalizer
  Generation:        2
  Resource Version:  22887
  UID:               23ee406d-9f8b-41b1-b723-2fb4e7167101
Spec:
  Destination:
    Create:     true
    Name:       db-secret
    Overwrite:  false
    Transformation:
  Mount:            database
  Path:             creds/readonly
  Renewal Percent:  67
  Vault Auth Ref:   vault-secrets-operator/example-jwt
Status:
  Conditions:
    Last Transition Time:  2026-10-07T16:24:53Z
    Message:               Secret synced
    Observed Generation:   2
    Reason:                Synced
    Status:                True
    Type:                  SecretSynced
    Last Transition Time:  2026-10-07T16:24:53Z
    Message:               Rollout restart triggered
    

Verificar que el secreto ha sido sincronizado

In [110]:

%%bash
kubectl wait --for=create secret/db-secret -n database --timeout=60s
kubectl get secret db-secret -n database -o yaml

secret/db-secret condition met
apiVersion: v1
data:
  _raw: eyJwYXNzd29yZCI6IkNpMndoc0ZnLVRxY0ttOWE4dmVkIiwidXNlcm5hbWUiOiJ2LWp3dC1zeXN0LXJlYWRvbmx5LWZ2ZVBwTTlLNlEyeGR5eGU2d0h2LTE3OTEzOTAyOTMifQ==
  password: Q2kyd2hzRmctVHFjS205YTh2ZWQ=
  username: di1qd3Qtc3lzdC1yZWFkb25seS1mdmVQcE05SzZRMnhkeXhlNndIdi0xNzkxMzkwMjkz
kind: Secret
metadata:
  creationTimestamp: "2026-10-07T16:24:53Z"
  labels:
    app.kubernetes.io/component: secret-sync
    app.kubernetes.io/managed-by: hashicorp-vso
    app.kubernetes.io/name: vault-secrets-operator
    secrets.hashicorp.com/vso-ownerRefUID: 23ee406d-9f8b-41b1-b723-2fb4e7167101
  name: db-secret
  namespace: database
  ownerReferences:
  - apiVersion: secrets.hashicorp.com/v1beta1
    kind: VaultDynamicSecret
    name: db-secret
    uid: 23ee406d-9f8b-41b1-b723-2fb4e7167101
  resourceVersion: "22879"
  uid: 26b5a4e4-675e-46cf-be30-35290061ac4d
type: Opaque


Ahora decodificando el base64

In [111]:
%%bash
echo "USERNAME: $(kubectl get secret db-secret -n database -o json | jq -r .data.username | base64 -d)"
echo "PASSWORD: $(kubectl get secret db-secret -n database -o json | jq -r .data.password | base64 -d)"

USERNAME: v-jwt-syst-readonly-fvePpM9K6Q2xdyxe6wHv-1791390293
PASSWORD: Ci2whsFg-TqcKm9a8ved


### Montemos el secreto en un POD

In [112]:
%%bash
cat > ${WORKDIR}/mypod_db.yaml <<EOF
apiVersion: v1
kind: Pod
metadata:
  name: mypoddb
  namespace: database
spec:
  containers:
  - name: mypod
    image: redis
    volumeMounts:
    - name: foo
      mountPath: "/etc/foo"
      readOnly: true
  volumes:
  - name: foo
    secret:
      secretName: db-secret
      optional: true
EOF


# Despliega el POD
kubectl apply -f ${WORKDIR}/mypod_db.yaml

pod/mypoddb created


Verifiquemos que el secreto se monta en el pod

In [113]:
%%bash
# Espera a que despliegue
sleep 10 
# Chequea secretos
kubectl exec mypoddb -n database  -- ls /etc/foo/
echo ""
echo "------"
kubectl exec mypoddb -n database -- cat /etc/foo/_raw

_raw
password
username

------
{"password":"Ci2whsFg-TqcKm9a8ved","username":"v-jwt-syst-readonly-fvePpM9K6Q2xdyxe6wHv-1791390293"}

Si esperamos 5 minutos (por la configuración del role) veremos un nuevo secreto en el volumen del POD

In [114]:
%%bash
sleep 200 
kubectl exec mypoddb -n database -- cat /etc/foo/_raw

{"password":"lYK6Dz5v5l48BkoW1t-k","username":"v-jwt-syst-readonly-16SkxATSTeYHgsmM9fI3-1791390469"}

### Transformations

In [115]:
%%bash

cat > ${WORKDIR}/dynamic_secret_tx.yaml <<EOF
---
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultDynamicSecret
metadata:
  namespace: database
  name: db-secret-tx
spec:
  vaultAuthRef: vault-secrets-operator/example-jwt
  mount: database
  path: creds/readonly
  destination:
    create: true
    name: db-secret-tx
    transformation:
      excludes:
       - .*
      templates:
        url:
          text: |
            {{- printf "postgresql://%s:%s@%s/postgres?sslmode=disable" (get .Secrets "username") (get .Secrets "password") -}}
EOF

kubectl apply -f ${WORKDIR}/dynamic_secret_tx.yaml


vaultdynamicsecret.secrets.hashicorp.com/db-secret-tx created


In [116]:
! kubectl describe  VaultDynamicSecret -n database db-secret-tx 

Name:         db-secret-tx
Namespace:    database
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultDynamicSecret
Metadata:
  Creation Timestamp:  2026-10-07T16:28:54Z
  Finalizers:
    vaultdynamicsecret.secrets.hashicorp.com/finalizer
  Generation:        1
  Resource Version:  23232
  UID:               49c29903-a777-421c-84a4-cda8ff7bb565
Spec:
  Destination:
    Create:     true
    Name:       db-secret-tx
    Overwrite:  false
    Transformation:
      Excludes:
        .*
      Templates:
        URL:
          Text:  {{- printf "postgresql://%s:%s@%s/postgres?sslmode=disable" (get .Secrets "username") (get .Secrets "password") -}}

  Mount:            database
  Path:             creds/readonly
  Renewal Percent:  67
  Vault Auth Ref:   vault-secrets-operator/example-jwt
Status:
  Conditions:
    Last Transition Time:  2026-10-07T16:28:54Z
    Message:               Secret synced
    Observed Generation:   1
    Reason:   

In [117]:
%%bash
# Print the resulting secret
kubectl get secret -n database db-secret-tx -o json | jq -r .data.url | base64 -d

postgresql://v-jwt-syst-readonly-7mIY3oijfoNCHcph4t3a-1791390534:RAWRSmUgfC4BQef-vJa5@%!s(MISSING)/postgres?sslmode=disable

### RolloutRestart

In [118]:
%%bash

cat > ${WORKDIR}/dynamic_secret2.yaml <<EOF
---
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultDynamicSecret
metadata:
  namespace: database
  name: db-secret-restart
spec:
  vaultAuthRef: vault-secrets-operator/example-jwt
  mount: database
  path: creds/readonly
  renewalPercent: 90            # Renew secret when 90% of its lease time has elapsed.
  revoke: true
  rolloutRestartTargets:        # Restart target deployments when secret data changes.
    - kind: Deployment
      name: mydbreader
  destination:
    create: true
    name: db-secret-restart
EOF

kubectl apply -f ${WORKDIR}/dynamic_secret2.yaml

vaultdynamicsecret.secrets.hashicorp.com/db-secret-restart created


In [119]:
! kubectl describe VaultDynamicSecret db-secret-restart -n database

Name:         db-secret-restart
Namespace:    database
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultDynamicSecret
Metadata:
  Creation Timestamp:  2026-10-07T16:28:55Z
  Finalizers:
    vaultdynamicsecret.secrets.hashicorp.com/finalizer
  Generation:        2
  Resource Version:  23242
  UID:               567c9c8f-2e52-4842-9dc7-b3849b076a93
Spec:
  Destination:
    Create:     true
    Name:       db-secret-restart
    Overwrite:  false
    Transformation:
  Mount:            database
  Path:             creds/readonly
  Renewal Percent:  90
  Revoke:           true
  Rollout Restart Targets:
    Kind:          Deployment
    Name:          mydbreader
  Vault Auth Ref:  vault-secrets-operator/example-jwt
Status:
  Conditions:
    Last Transition Time:  2026-10-07T16:28:55Z
    Message:               Secret synced
    Observed Generation:   2
    Reason:                Synced
    Status:                True
    Type:         

In [120]:
%%bash
kubectl get secret db-secret -n database -o json | jq -r .data.username | base64 -d
echo ""
kubectl get secret db-secret -n database -o json | jq -r .data.password | base64 -d
echo ""
echo "sleep 200 seconds to wait for renewal"
echo ""
sleep 200
kubectl get secret db-secret -n database -o json | jq -r .data.username | base64 -d
echo ""
kubectl get secret db-secret -n database -o json | jq -r .data.password | base64 -d

v-jwt-syst-readonly-16SkxATSTeYHgsmM9fI3-1791390469
lYK6Dz5v5l48BkoW1t-k
sleep 200 seconds to wait for renewal

v-jwt-syst-readonly-S86uVmtIXYHsHckl95on-1791390636
T8deOu9qDDNdcg-9Rel2

### Create Deployment to test restart on secret change

In [121]:
%%bash
cat > ${WORKDIR}/mydbreader.yaml <<EOF
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mydbreader
  namespace: database
spec:
  replicas: 2
  selector:
    matchLabels:
      app: mydbreader
  template:
    metadata:
      labels:
        app: mydbreader
    spec:
      containers:
      - name: nginx
        image: nginx:latest
        env:
        - name: USERNAME
          valueFrom:
            secretKeyRef:
              name: db-secret-restart
              key: username
        - name: PASSWORD
          valueFrom:
            secretKeyRef:
              name: db-secret-restart
              key: password
        command: ["/bin/sh"]
        args:
          - -c
          - |
            cat > /usr/share/nginx/html/index.html <<HTML
            <!DOCTYPE html>
            <html>
            <head>
              <title>Database Credentials</title>
              <style>
                body { font-family: Arial, sans-serif; margin: 40px; background-color: #f0f0f0; }
                .container { background-color: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }
                h1 { color: #333; }
                .credential { margin: 10px 0; padding: 10px; background-color: #e8f4f8; border-left: 4px solid #2196F3; }
                .label { font-weight: bold; color: #555; }
                .value { color: #2196F3; font-family: monospace; }
              </style>
            </head>
            <body>
              <div class="container">
                <h1>Database Credentials from Vault</h1>
                <div class="credential">
                  <span class="label">Username:</span>
                  <span class="value">\$USERNAME</span>
                </div>
                <div class="credential">
                  <span class="label">Password:</span>
                  <span class="value">\$PASSWORD</span>
                </div>
                <p><small>Pod: \$(hostname)</small></p>
              </div>
            </body>
            </html>
            HTML
            nginx -g 'daemon off;'
        ports:
        - containerPort: 80
          name: http
---
apiVersion: v1
kind: Service
metadata:
  name: mydbreader
  namespace: database
spec:
  selector:
    app: mydbreader
  ports:
  - port: 80
    targetPort: 80
    protocol: TCP
  type: ClusterIP
EOF

kubectl apply -f ${WORKDIR}/mydbreader.yaml

deployment.apps/mydbreader created
service/mydbreader created


In [122]:
! kubectl get deployments -n database mydbreader -o yaml 

apiVersion: apps/v1
kind: Deployment
metadata:
  annotations:
    deployment.kubernetes.io/revision: "1"
    kubectl.kubernetes.io/last-applied-configuration: |
      {"apiVersion":"apps/v1","kind":"Deployment","metadata":{"annotations":{},"name":"mydbreader","namespace":"database"},"spec":{"replicas":2,"selector":{"matchLabels":{"app":"mydbreader"}},"template":{"metadata":{"labels":{"app":"mydbreader"}},"spec":{"containers":[{"args":["-c","cat \u003e /usr/share/nginx/html/index.html \u003c\u003cHTML\n\u003c!DOCTYPE html\u003e\n\u003chtml\u003e\n\u003chead\u003e\n  \u003ctitle\u003eDatabase Credentials\u003c/title\u003e\n  \u003cstyle\u003e\n    body { font-family: Arial, sans-serif; margin: 40px; background-color: #f0f0f0; }\n    .container { background-color: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }\n    h1 { color: #333; }\n    .credential { margin: 10px 0; padding: 10px; background-color: #e8f4f8; border-left: 4px solid #2196F3; }\n    .lab

In [123]:
%%bash
echo "Waiting for deployment to be ready..."
kubectl wait --for=condition=available --timeout=200s deployment/mydbreader -n database
echo ""
echo "Deployment status:"
kubectl get deployment mydbreader -n database
echo ""
echo "Pods:"
kubectl get pods -n database -l app=mydbreader

Waiting for deployment to be ready...
deployment.apps/mydbreader condition met

Deployment status:
NAME         READY   UP-TO-DATE   AVAILABLE   AGE
mydbreader   2/2     2            2           4s

Pods:
NAME                          READY   STATUS    RESTARTS   AGE
mydbreader-769787b77d-76tjs   1/1     Running   0          4s
mydbreader-769787b77d-kq9x7   1/1     Running   0          4s


In [124]:
%%bash
echo "Service details:"
kubectl get svc mydbreader -n database
echo ""
echo "Testing the service (fetching the HTML page):"
kubectl run -it --rm curl-test --image=curlimages/curl --restart=Never -n database -- curl -s http://mydbreader.database.svc.cluster.local

Service details:
NAME         TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)   AGE
mydbreader   ClusterIP   10.96.213.178   <none>        80/TCP    4s

Testing the service (fetching the HTML page):
<!DOCTYPE html>
<html>
<head>
  <title>Database Credentials</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 40px; background-color: #f0f0f0; }
    .container { background-color: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }
    h1 { color: #333; }
    .credential { margin: 10px 0; padding: 10px; background-color: #e8f4f8; border-left: 4px solid #2196F3; }
    .label { font-weight: bold; color: #555; }
    .value { color: #2196F3; font-family: monospace; }
  </style>
</head>
<body>
  <div class="container">
    <h1>Database Credentials from Vault</h1>
    <div class="credential">
      <span class="label">Username:</span>
      <span class="value">v-jwt-syst-readonly-pWFeETaFDxdFVOkPk3Dw-1791390703</span>
    </div>
    <div class="

Verificar que el deployment se reinicia automáticamente cuando cambian las credenciales dinámicas

In [125]:
%%bash
echo "Current credentials displayed in the app:"
kubectl run -it --rm curl-test --image=curlimages/curl --restart=Never -n database -- curl -s http://mydbreader.database.svc.cluster.local | grep -A 2 "Username:\|Password:"
echo ""
echo "Current pod names (note the age):"
kubectl get pods -n database -l app=mydbreader
echo ""
echo "Waiting for credential renewal (this may take almost 200 seconds based on renewalPercent: 90)..."

Current credentials displayed in the app:
      <span class="label">Username:</span>
      <span class="value">v-jwt-syst-readonly-pWFeETaFDxdFVOkPk3Dw-1791390703</span>
    </div>
--
      <span class="label">Password:</span>
      <span class="value">zMvBc7szKnnzNwoCR5-4</span>
    </div>

Current pod names (note the age):
NAME                          READY   STATUS    RESTARTS   AGE
mydbreader-769787b77d-76tjs   1/1     Running   0          10s
mydbreader-769787b77d-kq9x7   1/1     Running   0          10s

Waiting for credential renewal (this may take almost 200 seconds based on renewalPercent: 90)...


In [126]:
%%bash
# sleep 200
echo "Current credentials displayed in the app:"
kubectl run -it --rm curl-test --image=curlimages/curl --restart=Never -n database -- curl -s http://mydbreader.database.svc.cluster.local | grep -A 2 "Username:\|Password:"
echo ""
echo "Current pod names (note the age):"
kubectl get pods -n database -l app=mydbreader
echo ""

Current credentials displayed in the app:
      <span class="label">Username:</span>
      <span class="value">v-jwt-syst-readonly-pWFeETaFDxdFVOkPk3Dw-1791390703</span>
    </div>
--
      <span class="label">Password:</span>
      <span class="value">zMvBc7szKnnzNwoCR5-4</span>
    </div>

Current pod names (note the age):
NAME                          READY   STATUS    RESTARTS   AGE
mydbreader-769787b77d-76tjs   1/1     Running   0          13s
mydbreader-769787b77d-kq9x7   1/1     Running   0          13s



In [127]:
! kubectl describe deploy -n database mydbreader

Name:                   mydbreader
Namespace:              database
CreationTimestamp:      Wed, 07 Oct 2026 18:32:15 +0200
Labels:                 <none>
Annotations:            deployment.kubernetes.io/revision: 1
Selector:               app=mydbreader
Replicas:               2 desired | 2 updated | 2 total | 2 available | 0 unavailable
StrategyType:           RollingUpdate
MinReadySeconds:        0
RollingUpdateStrategy:  25% max unavailable, 25% max surge
Pod Template:
  Labels:  app=mydbreader
  Containers:
   nginx:
    Image:      nginx:latest
    Port:       80/TCP (http)
    Host Port:  0/TCP (http)
    Command:
      /bin/sh
    Args:
      -c
      cat > /usr/share/nginx/html/index.html <<HTML
      <!DOCTYPE html>
      <html>
      <head>
        <title>Database Credentials</title>
        <style>
          body { font-family: Arial, sans-serif; margin: 40px; background-color: #f0f0f0; }
          .container { background-color: white; padding: 20px; border-radius: 8px; box

In [128]:
! kubectl describe VaultDynamicSecret db-secret-restart -n database

Name:         db-secret-restart
Namespace:    database
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultDynamicSecret
Metadata:
  Creation Timestamp:  2026-10-07T16:28:55Z
  Finalizers:
    vaultdynamicsecret.secrets.hashicorp.com/finalizer
  Generation:        2
  Resource Version:  23493
  UID:               567c9c8f-2e52-4842-9dc7-b3849b076a93
Spec:
  Destination:
    Create:     true
    Name:       db-secret-restart
    Overwrite:  false
    Transformation:
  Mount:            database
  Path:             creds/readonly
  Renewal Percent:  90
  Revoke:           true
  Rollout Restart Targets:
    Kind:          Deployment
    Name:          mydbreader
  Vault Auth Ref:  vault-secrets-operator/example-jwt
Status:
  Conditions:
    Last Transition Time:  2026-10-07T16:31:43Z
    Message:               Could not renew lease, err=lease renewal duration was truncated from 60s to 32s
    Observed Generation:   2
    Reason:       

## VaultPKISecret
https://developer.hashicorp.com/vault/docs/deploy/kubernetes/vso/api-reference#vaultpkisecret

### Modify permissions

In [129]:
%%bash
set -a; source ${WORKDIR}/config.env; set +a


vault policy write devk8s - <<EOF
path "kvv2/*" {
  capabilities = ["read", "list", "subscribe"]
  subscribe_event_types = ["*"]                     # https://developer.hashicorp.com/vault/docs/concepts/events#event-types
}
path "kv_v2/*" {
  capabilities = ["read", "list", "subscribe"]
  subscribe_event_types = ["*"]
}
path "sys/events/subscribe/*" {
  capabilities = ["read"]
}
path "database/creds/readonly" {
  capabilities = [ "read"]
}
path "pki_int/issue/example-dot-com" {
  capabilities = [ "update" ]
}
path "pki_int/revoke" {
  capabilities = [ "update" ]
}

EOF

Success! Uploaded policy: devk8s


In [130]:
! kubectl create ns pki

namespace/pki created


In [131]:
%%bash
cat > ${WORKDIR}/pki_secret.yaml <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultPKISecret
metadata:
  namespace: pki
  name: pki1
spec:
  vaultAuthRef: vault-secrets-operator/example-jwt
  mount: pki_int
  role: example-dot-com
  commonName: opus111.example.com
  altNames:
    - www.opus111.example.com
  revoke: true
  rolloutRestartTargets:        # Restart target deployments when secret data changes.
    - kind: Deployment
      name: mypkiapp
  format: pem
  expiryOffset: 30s             # Time before expiry to renew the certificate
  ttl: 300s                     # Duration of the certificate      
  destination:
    create: true
    name: pki1
EOF

kubectl apply -f ${WORKDIR}/pki_secret.yaml

vaultpkisecret.secrets.hashicorp.com/pki1 created


In [132]:
! kubectl describe VaultPKISecret pki1 -n pki

Name:         pki1
Namespace:    pki
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultPKISecret
Metadata:
  Creation Timestamp:  2026-10-07T16:32:29Z
  Generation:          1
  Resource Version:    23642
  UID:                 f5447450-1b6e-498a-b6e5-2c775c27a97a
Spec:
  Alt Names:
    www.opus111.example.com
  Common Name:  opus111.example.com
  Destination:
    Create:       true
    Name:         pki1
    Overwrite:    false
  Expiry Offset:  30s
  Format:         pem
  Mount:          pki_int
  Revoke:         true
  Role:           example-dot-com
  Rollout Restart Targets:
    Kind:          Deployment
    Name:          mypkiapp
  Ttl:             300s
  Vault Auth Ref:  vault-secrets-operator/example-jwt
Events:            <none>


In [134]:
%%bash
kubectl get secret pki1 -n pki -o yaml

apiVersion: v1
data:
  _raw: eyJhdXRob3JpdHlfa2V5X2lkIjoiNTM6NTA6NzY6N2U6YTU6ZDg6ODc6NDk6ZGY6YjI6OWM6ZTM6NzQ6NmM6MzQ6NWQ6N2M6ZGU6YjE6OGEiLCJjYV9jaGFpbiI6WyItLS0tLUJFR0lOIENFUlRJRklDQVRFLS0tLS1cbk1JSUVkakNDQTE2Z0F3SUJBZ0lVTUh3RVRIbm1IQ2piQ1JIWFNGQUNSSGVLOWNnd0RRWUpLb1pJaHZjTkFRRUxcbkJRQXdGakVVTUJJR0ExVUVBeE1MWlhoaGJYQnNaUzVqYjIwd0hoY05Nall4TURBM01UTTFPREkwV2hjTk16RXhcbk1EQTJNVE0xT0RVMFdqQXRNU3N3S1FZRFZRUURFeUpsZUdGdGNHeGxMbU52YlNCSmJuUmxjbTFsWkdsaGRHVWdcblFYVjBhRzl5YVhSNU1JSUJJakFOQmdrcWhraUc5dzBCQVFFRkFBT0NBUThBTUlJQkNnS0NBUUVBcmpHNmFkVUVcblFueWlXcjdCL1h1Y2R1Z1VTM3VFalFVc1BHZmxQeTFNSC9UcjFuenZIMTZMLzlvM2NpZVYxSFJWMzMzMDBYM3JcbjN3dVlIbTZ0MkJycmlQYUNjWUVOM1lMeEtDbDBDYm4rZWhiUVhNM3lHTndOQzlNck5iOCtIYm5YZ2FnbVcwc3Vcbmt5VGlYN2NLd3lrcDZkby91ZHBkNlJsd2tOOEtESm5EUjZFV1M5L09sRFVVU2xrY1hpYWd6RlM5Z2J6ZkNRWGhcbm1aV2cwcFNrWXdNMVF6SVJDYkowTVRFdWRqc2xUM3k0NnJYY3d5YWFaMmlOSEFmekcrT2hYTXY4cHp6V2NRcFZcbmZhRVRITGJ2SW03cGpuQ1FzZFpGUlptQTRRZEE2emlGR0JBSUtFK29TalBhdjFudVRtMEpmZDk5cUpUOU5SUDNcbmxzNUEyV1BiNTh

# Mount secret into Deployment

In [135]:
%%bash
cat > ${WORKDIR}/mypkiapp.yaml <<'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: nginx-config
  namespace: pki
data:
  nginx.conf: |
    events {
        worker_connections 1024;
    }
    http {
        server {
            listen 443 ssl;
            server_name opus111.example.com;
            
            ssl_certificate /etc/nginx/certs/tls.crt;
            ssl_certificate_key /etc/nginx/certs/tls.key;
            
            ssl_protocols TLSv1.2 TLSv1.3;
            ssl_ciphers HIGH:!aNULL:!MD5;
            
            location / {
                root /usr/share/nginx/html;
                index index.html;
            }
        }
    }
  index.html: |
    <!DOCTYPE html>
    <html>
    <head>
        <title>PKI Certificate Info</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                margin: 0;
                padding: 20px;
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                min-height: 100vh;
            }
            .container {
                max-width: 900px;
                margin: 0 auto;
                background: white;
                padding: 30px;
                border-radius: 10px;
                box-shadow: 0 10px 40px rgba(0,0,0,0.3);
            }
            h1 {
                color: #667eea;
                text-align: center;
                margin-bottom: 30px;
            }
            .cert-section {
                margin: 20px 0;
                padding: 20px;
                background: #f8f9fa;
                border-left: 4px solid #667eea;
                border-radius: 5px;
            }
            .cert-label {
                font-weight: bold;
                color: #555;
                margin-bottom: 5px;
            }
            .cert-value {
                color: #333;
                font-family: monospace;
                background: white;
                padding: 10px;
                border-radius: 3px;
                word-break: break-all;
            }
            .success {
                background: #d4edda;
                border-color: #28a745;
                color: #155724;
                padding: 15px;
                border-radius: 5px;
                margin: 20px 0;
            }
            .command {
                background: #2d3748;
                color: #68d391;
                padding: 15px;
                border-radius: 5px;
                margin: 20px 0;
                font-family: monospace;
                overflow-x: auto;
            }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Vault PKI Certificate Information</h1>
            <div class="success">
                This connection is secured with a certificate issued by HashiCorp Vault PKI
            </div>
            <div class="cert-section">
                <div class="cert-label">Common Name</div>
                <div class="cert-value">opus111.example.com</div>
            </div>
            <div class="cert-section">
                <div class="cert-label">Subject Alternative Names</div>
                <div class="cert-value">opus111.example.com, www.opus111.example.com</div>
            </div>
            <div class="cert-section">
                <div class="cert-label">Certificate TTL</div>
                <div class="cert-value">60 seconds (auto-rotates via VSO)</div>
            </div>
            <div class="cert-section">
                <div class="cert-label">Vault PKI Role</div>
                <div class="cert-value">example-dot-com</div>
            </div>
            <div class="cert-section">
                <div class="cert-label">Vault Mount Path</div>
                <div class="cert-value">pki_int</div>
            </div>
        </div>
    </body>
    </html>
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mypkiapp
  namespace: pki
spec:
  replicas: 5
  selector:
    matchLabels:
      app: mypkiapp
  template:
    metadata:
      labels:
        app: mypkiapp
    spec:
      containers:
      - name: nginx
        image: nginx:latest
        ports:
        - containerPort: 443
          name: https
        volumeMounts:
        - name: certs
          mountPath: /etc/nginx/certs
          readOnly: true
        - name: nginx-config
          mountPath: /etc/nginx/nginx.conf
          subPath: nginx.conf
          readOnly: true
        - name: html
          mountPath: /usr/share/nginx/html/index.html
          subPath: index.html
          readOnly: true
      volumes:
      - name: certs
        secret:
          secretName: pki1
          items:
          - key: certificate
            path: tls.crt
          - key: private_key
            path: tls.key
      - name: nginx-config
        configMap:
          name: nginx-config
      - name: html
        configMap:
          name: nginx-config
---
apiVersion: v1
kind: Service
metadata:
  name: mypkiapp
  namespace: pki
spec:
  selector:
    app: mypkiapp
  ports:
  - port: 443
    targetPort: 443
    protocol: TCP
    name: https
  type: ClusterIP
EOF

kubectl apply -f ${WORKDIR}/mypkiapp.yaml

configmap/nginx-config created
deployment.apps/mypkiapp created
service/mypkiapp created


In [136]:
%%bash
echo "Waiting for deployment to be ready..."
kubectl wait --for=condition=available --timeout=60s deployment/mypkiapp -n pki
echo ""
echo "Deployment status:"
kubectl get deployment mypkiapp -n pki
echo ""
echo "Pods:"
kubectl get pods -n pki -l app=mypkiapp
echo ""
echo "Service:"
kubectl get svc mypkiapp -n pki

Waiting for deployment to be ready...
deployment.apps/mypkiapp condition met

Deployment status:
NAME       READY   UP-TO-DATE   AVAILABLE   AGE
mypkiapp   4/5     5            4           4s

Pods:
NAME                       READY   STATUS              RESTARTS   AGE
mypkiapp-f8694f76c-b5xhx   1/1     Running             0          5s
mypkiapp-f8694f76c-g7h76   0/1     ContainerCreating   0          5s
mypkiapp-f8694f76c-ghlfl   1/1     Running             0          5s
mypkiapp-f8694f76c-lgn4z   1/1     Running             0          5s
mypkiapp-f8694f76c-wb5gf   1/1     Running             0          5s

Service:
NAME       TYPE        CLUSTER-IP       EXTERNAL-IP   PORT(S)   AGE
mypkiapp   ClusterIP   10.103.194.124   <none>        443/TCP   5s


Verificar el certificado montado en el deployment

In [137]:
%%bash
echo "Certificate details from the mounted volume:"
kubectl exec -n pki deploy/mypkiapp -- openssl x509 -in /etc/nginx/certs/tls.crt -text -noout | grep -A 5 "Subject:\|Issuer:\|Validity\|DNS:"

Certificate details from the mounted volume:
        Issuer: CN=example.com Intermediate Authority
        Validity
            Not Before: Oct  7 16:32:00 2026 GMT
            Not After : Oct  7 16:37:30 2026 GMT
        Subject: CN=opus111.example.com
        Subject Public Key Info:
            Public Key Algorithm: rsaEncryption
                Public-Key: (2048 bit)
                Modulus:
                    00:fb:b1:19:d9:9f:08:cb:f6:a2:93:36:f1:46:cb:
--
                DNS:opus111.example.com, DNS:www.opus111.example.com
            X509v3 CRL Distribution Points: 
                Full Name:
                  URI:https://vault.vault.svc.cluster.local:8200/v1/pki_int/issuer/977d21a7-f977-9690-17c0-e9d8694245cc/crl/der

    Signature Algorithm: sha256WithRSAEncryption


Probar la conexión TLS al servicio

In [138]:
%%bash

echo "Testing HTTPS connection to the service:"
kubectl run -it --rm curl-test --image=curlimages/curl --restart=Never -n pki -- curl -k https://mypkiapp.pki.svc.cluster.local | head -50

Testing HTTPS connection to the service:
<!DOCTYPE html>
<html>
<head>
    <title>PKI Certificate Info</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
        }
        .container {
            max-width: 900px;
            margin: 0 auto;
            background: white;
            padding: 30px;
            border-radius: 10px;
            box-shadow: 0 10px 40px rgba(0,0,0,0.3);
        }
        h1 {
            color: #667eea;
            text-align: center;
            margin-bottom: 30px;
        }
        .cert-section {
            margin: 20px 0;
            padding: 20px;
            background: #f8f9fa;
            border-left: 4px solid #667eea;
            border-radius: 5px;
        }
        .cert-label {
            font-weight: bold;
            color: #555;
            margin-bottom: 

Verificar que el certificado se renueva automáticamente (espera 4 mins y medio por el TTL)

In [139]:
%%bash
echo "Current certificate serial number:"
kubectl exec -n pki deploy/mypkiapp -- openssl x509 -in /etc/nginx/certs/tls.crt -serial -noout
echo ""
echo "Current pods age:"
kubectl get pods -n pki -l app=mypkiapp
echo ""
echo "Waiting 270 seconds for certificate renewal and rollout restart..."
sleep 270
echo ""
echo "New certificate serial number (should be different):"
kubectl exec -n pki deploy/mypkiapp -- openssl x509 -in /etc/nginx/certs/tls.crt -serial -noout
echo ""
echo "New pods (check age - should be recently restarted):"
kubectl get pods -n pki -l app=mypkiapp

Current certificate serial number:
serial=548CF4A349F5E63041B391296D1E5C5A6DBC5133

Current pods age:
NAME                       READY   STATUS    RESTARTS   AGE
mypkiapp-f8694f76c-b5xhx   1/1     Running   0          21s
mypkiapp-f8694f76c-g7h76   1/1     Running   0          21s
mypkiapp-f8694f76c-ghlfl   1/1     Running   0          21s
mypkiapp-f8694f76c-lgn4z   1/1     Running   0          21s
mypkiapp-f8694f76c-wb5gf   1/1     Running   0          21s

Waiting 270 seconds for certificate renewal and rollout restart...

New certificate serial number (should be different):
serial=285F010BA9E954FF0CB8135A5A3D479C9DCE8B4C

New pods (check age - should be recently restarted):
NAME                        READY   STATUS    RESTARTS   AGE
mypkiapp-5f7b66cdc7-2j8fs   1/1     Running   0          117s
mypkiapp-5f7b66cdc7-2jjml   1/1     Running   0          117s
mypkiapp-5f7b66cdc7-45fkw   1/1     Running   0          115s
mypkiapp-5f7b66cdc7-78nxg   1/1     Running   0          117s
mypki

In [140]:
! kubectl describe deploy -n pki mypkiapp

Name:                   mypkiapp
Namespace:              pki
CreationTimestamp:      Wed, 07 Oct 2026 18:34:14 +0200
Labels:                 <none>
Annotations:            deployment.kubernetes.io/revision: 2
Selector:               app=mypkiapp
Replicas:               5 desired | 5 updated | 5 total | 5 available | 0 unavailable
StrategyType:           RollingUpdate
MinReadySeconds:        0
RollingUpdateStrategy:  25% max unavailable, 25% max surge
Pod Template:
  Labels:       app=mypkiapp
  Annotations:  vso.secrets.hashicorp.com/restartedAt: 2026-10-07T16:37:07Z
  Containers:
   nginx:
    Image:        nginx:latest
    Port:         443/TCP (https)
    Host Port:    0/TCP (https)
    Environment:  <none>
    Mounts:
      /etc/nginx/certs from certs (ro)
      /etc/nginx/nginx.conf from nginx-config (ro,path="nginx.conf")
      /usr/share/nginx/html/index.html from html (ro,path="index.html")
  Volumes:
   certs:
    Type:        Secret (a volume populated by a Secret)
    Secret

## Create forwarding port to access deployment

# Clean UP

Elimina los recursos creados en este cuaderno. Primero los CR de VSO, para que el operador retire los secretos, y después los namespaces, el operador y la configuración JWT de Vault.

In [41]:
%%bash
# Si el operador ya no está, el finalizer deja el delete en espera y kubectl
# termina con código 1. Si el CRD ya no existe, no hay nada que borrar.
delete_crs() {
  kind="$1"
  ns="$2"
  shift 2
  kubectl api-resources --api-group=secrets.hashicorp.com -o name 2>/dev/null | grep -q "^${kind}" || return 0
  for name in "$@"; do
    kubectl patch "$kind" "$name" -n "$ns" --type=merge -p '{"metadata":{"finalizers":null}}' 2>/dev/null || true
  done
  kubectl delete "$kind" "$@" -n "$ns" --ignore-not-found --timeout=30s
}

delete_crs vaultstaticsecret static-jwt example
delete_crs vaultstaticsecret static example2 example-event
delete_crs vaultdynamicsecret database db-secret db-secret-tx db-secret-restart
delete_crs vaultpkisecret pki pki1

In [42]:
%%bash
kubectl delete ns static-jwt static database pki --ignore-not-found

namespace "static-jwt" deleted


In [43]:
%%bash
kubectl delete clusterrolebinding oidc-reviewer --ignore-not-found
kubectl delete vaultauth example-jwt -n vault-secrets-operator --ignore-not-found
kubectl delete vaultconnection example -n vault-secrets-operator --ignore-not-found

helm uninstall vault-secrets-operator -n vault-secrets-operator
kubectl delete ns vault-secrets-operator --ignore-not-found

clusterrolebinding.rbac.authorization.k8s.io "oidc-reviewer" deleted


error: the server doesn't have a resource type "vaultauth"
error: the server doesn't have a resource type "vaultconnection"
Error: uninstall: Release not loaded: vault-secrets-operator: release: not found


Helm no borra los CRD. Se eliminan a continuación.

In [44]:
%%bash
kubectl delete crd \
  csisecrets.secrets.hashicorp.com \
  hcpauths.secrets.hashicorp.com \
  hcpvaultsecretsapps.secrets.hashicorp.com \
  secrettransformations.secrets.hashicorp.com \
  vaultauthglobals.secrets.hashicorp.com \
  vaultauths.secrets.hashicorp.com \
  vaultconnections.secrets.hashicorp.com \
  vaultdynamicsecrets.secrets.hashicorp.com \
  vaultpkisecrets.secrets.hashicorp.com \
  vaultstaticsecrets.secrets.hashicorp.com \
  --ignore-not-found

In [45]:
%%bash
export WORKDIR="${WORKDIR:-/tmp/vault}"
set -a; source ${WORKDIR}/config.env; set +a

vault auth disable jwt
vault policy delete devk8s
vault secrets disable kv_v2

Success! Disabled the auth method (if it existed) at: jwt/
Success! Deleted policy: devk8s
Success! Disabled the secrets engine (if it existed) at: kv_v2/


In [46]:
%%bash
export WORKDIR="${WORKDIR:-/tmp/vault}"
rm -f \
  ${WORKDIR}/k8s-ca.pem \
  ${WORKDIR}/jwt-role.json \
  ${WORKDIR}/vso_crd.yaml \
  ${WORKDIR}/vaultauth_crd.yaml \
  ${WORKDIR}/static_secret.yaml \
  ${WORKDIR}/static_secret2.yaml \
  ${WORKDIR}/static_secret_event.yaml \
  ${WORKDIR}/mypod.yaml \
  ${WORKDIR}/mydeployment.yaml \
  ${WORKDIR}/mydeployment2.yaml \
  ${WORKDIR}/dynamic_secret.yaml \
  ${WORKDIR}/dynamic_secret_tx.yaml \
  ${WORKDIR}/dynamic_secret2.yaml \
  ${WORKDIR}/mypod_db.yaml \
  ${WORKDIR}/mydbreader.yaml \
  ${WORKDIR}/pki_secret.yaml \
  ${WORKDIR}/mypkiapp.yaml